# Homework: English-to-Armenian Seq2Seq Translation with Attention

In this assignment, you will implement a **Dot-Product Attention** mechanism to bridge an Encoder-Decoder Recurrent Neural Network (RNN) architecture for Machine Translation.

You are provided with:
1. Data loading and tokenization using the OPUS-100 English-Armenian dataset.
2. A simple `EncoderRNN` module.
3. The training loop scaffolding.

**Your Task:**
Complete the linear algebra operations inside the `DotProductAttention` and `DecoderAttnRNN` modules, and fill in the missing PyTorch optimization steps in the training loop.

In [1]:
!pip install -q datasets

import math
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Special Tokens
PAD_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
UNK_TOKEN = 3

class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2index = {"<PAD>": PAD_TOKEN, "<SOS>": SOS_TOKEN, "<EOS>": EOS_TOKEN, "<UNK>": UNK_TOKEN}
        self.index2word = {PAD_TOKEN: "<PAD>", SOS_TOKEN: "<SOS>", EOS_TOKEN: "<EOS>", UNK_TOKEN: "<UNK>"}
        self.word2count = {}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.strip().split():
            self.add_word(word.lower())

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

def tokenize_and_build_vocab(dataset_subset, max_samples=15000):
    src_vocab = Vocabulary("English")
    tgt_vocab = Vocabulary("Armenian")
    pairs = []

    for item in dataset_subset:
        en_text = item['translation']['en'].strip().lower()
        hy_text = item['translation']['hy'].strip().lower()

        # Keep short sentences for fast Colab training
        if 2 <= len(en_text.split()) <= 15 and 2 <= len(hy_text.split()) <= 15:
            src_vocab.add_sentence(en_text)
            tgt_vocab.add_sentence(hy_text)
            pairs.append((en_text, hy_text))

        if len(pairs) >= max_samples:
            break

    return src_vocab, tgt_vocab, pairs

print("Loading OPUS-100 dataset...")
raw_dataset = load_dataset("Helsinki-NLP/opus-100", "en-hy", split="train")
src_vocab, tgt_vocab, pairs = tokenize_and_build_vocab(raw_dataset, max_samples=15000)

print(f"Dataset Loaded! Pair count: {len(pairs)}")
print(f"English Vocab Size: {src_vocab.n_words}")
print(f"Armenian Vocab Size: {tgt_vocab.n_words}")

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab, max_len=18):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def tensor_from_sentence(self, vocab, sentence):
        indexes = [vocab.word2index.get(w, UNK_TOKEN) for w in sentence.split()]
        indexes.append(EOS_TOKEN)
        # Pad sequence
        if len(indexes) < self.max_len:
            indexes += [PAD_TOKEN] * (self.max_len - len(indexes))
        return torch.tensor(indexes[:self.max_len], dtype=torch.long)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src_tensor = self.tensor_from_sentence(self.src_vocab, src_text)
        tgt_tensor = self.tensor_from_sentence(self.tgt_vocab, tgt_text)
        return src_tensor, tgt_tensor

BATCH_SIZE = 64
dataset = TranslationDataset(pairs, src_vocab, tgt_vocab)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

Using device: cpu
Loading OPUS-100 dataset...


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-hy/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  423kB            

en-hy/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7059 [00:00<?, ? examples/s]

Dataset Loaded! Pair count: 4903
English Vocab Size: 6022
Armenian Vocab Size: 8354


### Encoder Architecture (Provided)

The Encoder processes the input English sequence using a continuous embedding layer and a basic single-layer Recurrent Neural Network (RNN).

In [2]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size, padding_idx=PAD_TOKEN)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)

    def forward(self, input_seq):
        # input_seq shape: (batch_size, seq_len)
        embedded = self.embedding(input_seq) # (batch_size, seq_len, hidden_size)
        outputs, hidden = self.rnn(embedded)
        # outputs shape: (batch_size, seq_len, hidden_size)
        # hidden shape: (1, batch_size, hidden_size)
        return outputs, hidden

### Task 1: Implement Dot-Product Attention

Complete the missing linear algebra steps in the `DotProductAttention` block:
1. Compute the raw attention scores via matrix multiplication between the decoder hidden state and encoder outputs.
2. Apply softmax along the sequence dimension to obtain normalized attention weights.
3. Compute the context vector as a weighted sum of encoder outputs.

In [3]:
class DotProductAttention(nn.Module):
    def __init__(self):
        super(DotProductAttention, self).__init__()

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: Current hidden state of the decoder -> shape (batch_size, 1, hidden_size)
        encoder_outputs: Output hidden states of encoder -> shape (batch_size, seq_len, hidden_size)
        """
        # TODO 1: Compute attention scores using batch matrix multiplication (torch.bmm).
        # Hint: You need to multiply (batch_size, 1, hidden_size) by (batch_size, hidden_size, seq_len).
        scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))
        # scores shape: (batch_size, 1, seq_len)

        # TODO 2: Normalize the scores using softmax along the sequence length dimension (dim=-1)
        attn_weights = F.softmax(scores, dim=-1)
        # attn_weights shape: (batch_size, 1, seq_len)

        # TODO 3: Compute context vector by weighting the encoder_outputs with attn_weights
        # Hint: Batch matrix multiply (batch_size, 1, seq_len) by (batch_size, seq_len, hidden_size)
        context = torch.bmm(attn_weights, encoder_outputs)
        # context shape: (batch_size, 1, hidden_size)

        return context, attn_weights

### Task 2: Implement the Attention Decoder

Complete the `DecoderAttnRNN` block:
1. Call your `DotProductAttention` module using the previous hidden state and `encoder_outputs`.
2. Concatenate the embedded input token vector and the context vector along the feature dimension (`dim=-1`).
3. Pass the concatenated representation into the RNN layer, then project to the target vocabulary size using the linear layer.

In [4]:
class DecoderAttnRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderAttnRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size, padding_idx=PAD_TOKEN)
        self.attention = DotProductAttention()
        self.rnn = nn.RNN(hidden_size * 2, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, word_input, hidden, encoder_outputs):
        """
        word_input: (batch_size, 1)
        hidden: (1, batch_size, hidden_size)
        encoder_outputs: (batch_size, seq_len, hidden_size)
        """
        embedded = self.embedding(word_input) # (batch_size, 1, hidden_size)

        # Reshape decoder hidden state to (batch_size, 1, hidden_size) for attention calculation
        hidden_reshaped = hidden.permute(1, 0, 2)

        # TODO 4: Calculate context and attention weights using your attention layer
        context, attn_weights = self.attention(hidden_reshaped, encoder_outputs)

        # TODO 5: Concatenate embedded input and context vector along feature dimension (dim=2 or dim=-1)
        rnn_input = torch.cat((embedded, context), dim=-1)

        # TODO 6: Pass rnn_input through self.rnn using hidden as the initial state
        rnn_output, hidden = self.rnn(rnn_input, hidden)

        # TODO 7: Pass rnn_output through self.out linear layer to get predictions (batch_size, 1, output_size)
        predictions = self.out(rnn_output)

        return predictions, hidden, attn_weights

### Task 3: Complete the Training Loop

Complete the training step:
1. Zero the gradients of both optimizers.
2. Calculate the cross-entropy loss between predicted logits and target tokens.
3. Perform backward propagation and step both optimizers.

In [5]:
HIDDEN_SIZE = 256
LEARNING_RATE = 0.001
EPOCHS = 10

encoder = EncoderRNN(src_vocab.n_words, HIDDEN_SIZE).to(device)
decoder = DecoderAttnRNN(HIDDEN_SIZE, tgt_vocab.n_words).to(device)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr=LEARNING_RATE)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

def train_epoch(dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    encoder.train()
    decoder.train()
    total_loss = 0

    for src_batch, tgt_batch in dataloader:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)
        batch_size = src_batch.size(0)

        # TODO 8: Zero gradients for both optimizers
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(src_batch)

        decoder_input = torch.tensor([[SOS_TOKEN]] * batch_size, device=device) # (batch_size, 1)
        decoder_hidden = encoder_hidden

        loss = 0

        # Teacher forcing: Feed actual target token as next input
        for t in range(tgt_batch.size(1) - 1):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)

            target_step = tgt_batch[:, t + 1] # Target word for step t

            # TODO 9: Calculate step loss using criterion and accumulate into 'loss' variable
            # Hint: decoder_output is (batch_size, 1, tgt_vocab_size), flatten it to (batch_size, tgt_vocab_size)
            step_loss = criterion(decoder_output.squeeze(1), target_step)
            loss += step_loss

            decoder_input = tgt_batch[:, t + 1].unsqueeze(1) # Next input is current target

        # TODO 10: Perform backpropagation on total accumulated loss
        loss.backward()

        # TODO 11: Step both optimizers
        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item() / (tgt_batch.size(1) - 1)

    return total_loss / len(dataloader)

# Run Training Loop
print("Starting Training...")
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {loss:.4f}")

Starting Training...
Epoch 1/10 - Loss: nan
Epoch 2/10 - Loss: nan
Epoch 3/10 - Loss: nan
Epoch 4/10 - Loss: nan
Epoch 5/10 - Loss: nan
Epoch 6/10 - Loss: nan
Epoch 7/10 - Loss: nan
Epoch 8/10 - Loss: nan
Epoch 9/10 - Loss: nan
Epoch 10/10 - Loss: nan


### Task 4: Evaluation & Inference

Translate sample English sentences using autoregressive greedy decoding.

In [6]:
def translate(sentence, encoder, decoder, src_vocab, tgt_vocab, max_len=18):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        tokens = [src_vocab.word2index.get(w, UNK_TOKEN) for w in sentence.lower().split()]
        tokens.append(EOS_TOKEN)
        if len(tokens) < max_len:
            tokens += [PAD_TOKEN] * (max_len - len(tokens))

        src_tensor = torch.tensor(tokens[:max_len], dtype=torch.long, device=device).unsqueeze(0)

        encoder_outputs, encoder_hidden = encoder(src_tensor)

        decoder_input = torch.tensor([[SOS_TOKEN]], device=device)
        decoder_hidden = encoder_hidden

        translated_words = []

        for _ in range(max_len):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topi = decoder_output.argmax(dim=-1).item()

            if topi == EOS_TOKEN:
                break

            translated_words.append(tgt_vocab.index2word.get(topi, "<UNK>"))
            decoder_input = torch.tensor([[topi]], device=device)

        return " ".join(translated_words)

# Test translation on a training sample
test_sentence = pairs[0][0]
print(f"\nSource (EN): {test_sentence}")
print(f"Target (HY): {pairs[0][1]}")
print(f"Model Prediction: {translate(test_sentence, encoder, decoder, src_vocab, tgt_vocab)}")


Source (EN): nurse, commend me to thy lady and mistress, i protest unto thee.
Target (HY): դայակ, քո տիրուհուն իմ կողմից ողջույն ուղարկիր, և իմացիր, որ ասածներիդ դեմ ես բողոքում եմ:
Model Prediction: national month 5 - պարոն կատադրոյֆե՞։
